## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

In [12]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [27]:
from statsmodels.stats.proportion import proportion_effectsize

retention_standard = 0.19
retention_new = 0.20

effect_size = proportion_effectsize(retention_new, retention_standard)
print(effect_size)

0.025241594409087353


Отримали дуже малий effect size, який складає 0,025, для того, щоб це помітити потрібна дуже велика вибірка

In [28]:
import math
from statsmodels.stats import power as sms

required_n = math.ceil(sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
))

print(f'Потрібно сумарно {required_n*2} користувачів для проведення A/B тестування')

Потрібно сумарно 49276 користувачів для проведення A/B тестування


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [29]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/data/home_tasks/cookie_cats.csv')
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [30]:
retention_by_version = df.groupby('version')['retention_7'].mean()
print('Середнє значення показника утримання на 7 день за групами')
print(retention_by_version)

Середнє значення показника утримання на 7 день за групами
version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64


Гіпотеза
Н0: p1=p2
Ha: р2>p1
де p1 - частка утримання гравців у контрольній групі gate_30
   p2 - частка утримання гравців у тестовій групі gate_40
Отже це правосторонній тест, оскільки при альтернативній гіпотезі вважаємо, що частка утримання гравців у тестовій групі gate_40 більша.


3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [35]:
control = df[df['version'] == 'gate_30']
treatment = df[df['version'] == 'gate_40']
control_success = control['retention_7'].sum()
control_total = len(control)
treatment_success = treatment['retention_7'].sum()
treatment_total = len(treatment)

In [36]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

count = [treatment_success, control_success]
nobs = [treatment_total, control_total]

z_stat, p_value = proportions_ztest(count, nobs, alternative='larger')

lower_con, upper_con = proportion_confint(control_success, control_total, alpha=0.05)
lower_treat, upper_treat = proportion_confint(treatment_success, treatment_total, alpha=0.05)

print(f'z statistic: {z_stat:.2f}')
print(f'p-value: {p_value:.3f}')
print(f'Довірчий інтервал 95% для групи control: [{lower_con:.3f}, {upper_con:.3f}]')
print(f'Довірчий інтервал 95% для групи treatment: [{lower_treat:.3f}, {upper_treat:.3f}]')

z statistic: -3.16
p-value: 0.999
Довірчий інтервал 95% для групи control: [0.187, 0.194]
Довірчий інтервал 95% для групи treatment: [0.178, 0.186]


Висновок:
1. Оскільки p-value > α (0.999 > 0.05), ми не відхиляємо нульову гіпотезу. Немає статистичних підстав стверджувати, що версія gate_40 має кращий показник retention_7, ніж gate_30.
2. z-statistic = −3.16 показує, що рівень утримання користувачів у gate_40 нижчий, ніж у gate_30.
3. Довірчі інтервали для двох груп не перетинаються: для control вони вищі, ніж для treatment. Це вказує на наявність різниці між групами, однак статистичний тест не підтвердив її значущість на рівні α = 0.05.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


H₀: версія гри та утримання незалежні
Hₐ: версія гри та утримання залежать

In [37]:
contingency_table = pd.crosstab(df['version'], df['retention_7'])
print(contingency_table)

retention_7  False  True 
version                  
gate_30      36198   8502
gate_40      37210   8279


In [40]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"χ² = {chi2:.3f}")
print(f"p-value = {p_value:.5f}")
print(f"Ступені свободи = {dof}")
print("Очікувані частоти:\n", expected)



χ² = 9.959
p-value = 0.00160
Ступені свободи = 1
Очікувані частоти:
 [[36382.90257127  8317.09742873]
 [37025.09742873  8463.90257127]]


In [41]:
alpha = 0.05
if p_value < alpha:
    print("Відхиляємо H0 → є залежність")
else:
    print("Не відхиляємо H0 → залежності не виявлено")

Відхиляємо H0 → є залежність
